# 1. Load, gộp mapping giữa BKG_entity2Id.json và node2id.json

In [1]:
import pandas as pd
import json
import requests

In [2]:
train_df = 'dataset/MUDIv2_train.csv'
val_df = 'dataset/MUDIv2_val.csv'
test_df = 'dataset/MUDIv2_test.csv'

In [3]:
m1 = requests.get(
    "https://raw.githubusercontent.com/LARS-research/KnowDDI/main/raw_data/Drugbank/node2id.json"
).json()

m2 = requests.get(
    "https://raw.githubusercontent.com/LARS-research/KnowDDI/main/raw_data/Drugbank/BKG_entity2Id.json"
).json()
m2_cleaned = {k.replace("Compound::", ""): v for k, v in m2.items()}
mapping = {**m2_cleaned, **m1}
print("Total entities:", len(mapping))

Total entities: 34124


In [4]:
with open("raw_data/Drugbank/node2id.json", "r") as f:
    node2id = json.load(f)

node2id_cleaned = {(("Compound::" + k)): v for k, v in node2id.items()}
print("Total entities:", len(node2id_cleaned))


Total entities: 1710


In [5]:
with open("raw_data/Drugbank/BKG_entity2Id.json", "r") as f:
    entity2id = json.load(f)
print("Total entities:", len(entity2id))

Total entities: 32414


In [6]:
old_mapping = {**node2id_cleaned, **entity2id}
list(old_mapping.items())[:10]

[('Compound::DB04571', 0),
 ('Compound::DB00460', 1),
 ('Compound::DB00855', 2),
 ('Compound::DB09536', 3),
 ('Compound::DB01600', 4),
 ('Compound::DB09000', 5),
 ('Compound::DB11630', 6),
 ('Compound::DB00553', 7),
 ('Compound::DB06261', 8),
 ('Compound::DB01878', 9)]

In [7]:
old_id2mapping = {v: k for k, v in old_mapping.items()}

## 1.2. New mapping

In [8]:
with open("dataset/valid_drugs.txt", "r") as f:
    valid_drugs = f.read().splitlines()
print("Total valid drugs:", len(valid_drugs))

Total valid drugs: 1295


In [9]:
# Tạo mapping drug -> id
drug2id = {drug.strip(): idx for idx, drug in enumerate(valid_drugs)}

# Nếu muốn id -> drug
id2drug = {idx: drug for drug, idx in drug2id.items()}
len(drug2id)

1295

In [10]:
new_mapping = {**drug2id}
len(new_mapping)

1295

In [11]:
bkg = pd.read_csv(
    "raw_data/Drugbank/BKG_file.txt",
    delim_whitespace=True,
    header=None
)

print(bkg.shape)
print(bkg.head())
value_set = set(bkg[0]).union(set(bkg[1]))
len(value_set)

C:\Users\Admin\AppData\Local\Temp\ipykernel_28084\3885242154.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  bkg = pd.read_csv(


(1690693, 3)
      0     1  2
0  1710  1711  0
1  1712  1713  0
2  1714  1715  0
3  1716  1717  0
4  1718  1719  0


33765

In [12]:
new2old_mapping = {}
out_kg = set()

In [13]:
list(new_mapping.items())[:10]

[('Compound::DB08798', 0),
 ('Compound::DB00713', 1),
 ('Compound::DB00915', 2),
 ('Compound::DB01061', 3),
 ('Compound::DB01619', 4),
 ('Compound::DB06196', 5),
 ('Compound::DB01192', 6),
 ('Compound::DB04938', 7),
 ('Compound::DB06782', 8),
 ('Compound::DB00184', 9)]

In [14]:
# Giả sử các biến có sẵn:
# old_mapping: dict node_name -> old_id
# new_mapping: dict node_name -> new_id (một số đã có trước)
# value_set: set of node_name (từ BKG)
# out_kg: set()  # nodes not in value_set

old2new = {}  # sẽ map old_id -> new_id

# bắt đầu cấp id mới từ next_id (giữ nguyên các id hiện có trong new_mapping)
if new_mapping:
    next_id = max(new_mapping.values()) + 1
else:
    next_id = 0

# 1) xử lý các node trong old_mapping
for node, old_id in old_mapping.items():
    # nếu node đã có trong new_mapping thì chỉ nối nối lại mapping
    if node in new_mapping:
        old2new[old_id] = new_mapping[node]
        continue

    # nếu node không nằm trong value_set => đưa vào out_kg (không cấp id ngay)
    if old_id not in value_set:
        out_kg.add(node)
        continue

    # nếu node cần được thêm vào new_mapping:
    new_mapping[node] = next_id
    old2new[old_id] = next_id
    next_id += 1

# 2) nếu bạn vẫn muốn cấp id cho những node trong out_kg:
for node in out_kg:
    # tránh cấp id nếu đã vô tình có
    if node in new_mapping:
        continue
    new_mapping[node] = next_id
    # đảm bảo old_mapping có node (nếu không có thì skip)
    if node in old_mapping:
        old2new[ old_mapping[node] ] = next_id
    next_id += 1

# kết quả:
print("Final new_mapping size:", len(new_mapping))
# old2new maps old_id -> new_id


Final new_mapping size: 34124


In [15]:
new_mapping["Compound::DB00316"]

719

# 2. Mapping Drug1, Drug2 sang Idx

In [16]:
train = pd.read_csv(train_df)
val   = pd.read_csv(val_df)
test  = pd.read_csv(test_df)

for df in [train, val, test]:
    # Loại bỏ 'Compound::' nếu có
    df["Drug1_clean"] = df["Drug1"]
    df["Drug2_clean"] = df["Drug2"]
    
    # Ánh xạ sang ID
    df["drug1_id"] = df["Drug1_clean"].map(new_mapping)
    df["drug2_id"] = df["Drug2_clean"].map(new_mapping)

C:\Users\Admin\AppData\Local\Temp\ipykernel_28084\2405321137.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  test  = pd.read_csv(test_df)


In [17]:
for name, df in zip(["train","val","test"], [train, val, test]):
    total_pairs = len(df)

    miss_d1 = df["drug1_id"].isna().sum()
    miss_d2 = df["drug2_id"].isna().sum()

    mapped_d1 = total_pairs - miss_d1
    mapped_d2 = total_pairs - miss_d2

    print(f"=== {name.upper()} ===")
    print("Total pairs:", total_pairs)
    print("Drug1  mapped:", mapped_d1, "| missing:", miss_d1)
    print("Drug2  mapped:", mapped_d2, "| missing:", miss_d2)

    # số cặp được map cả 2 thuốc
    both_ok = df["drug1_id"].notna() & df["drug2_id"].notna()
    print("Pairs fully mapped:", both_ok.sum())
    print("Pairs with missing mapping:", total_pairs - both_ok.sum())
    print()


=== TRAIN ===
Total pairs: 346859
Drug1  mapped: 346859 | missing: 0
Drug2  mapped: 346859 | missing: 0
Pairs fully mapped: 346859
Pairs with missing mapping: 0

=== VAL ===
Total pairs: 113886
Drug1  mapped: 113886 | missing: 0
Drug2  mapped: 113886 | missing: 0
Pairs fully mapped: 113886
Pairs with missing mapping: 0

=== TEST ===
Total pairs: 343404
Drug1  mapped: 343404 | missing: 0
Drug2  mapped: 343404 | missing: 0
Pairs fully mapped: 343404
Pairs with missing mapping: 0



In [18]:
train

,Drug1,Pharmacodynamics,Pharmacokinetics,Drug2,Drug1_clean,Drug2_clean,drug1_id,drug2_id
0,Compound::DB00842,Synergism,Excretion,Compound::DB00316,Compound::DB00842,Compound::DB00316,361,719
1,Compound::DB00495,Synergism,Unknown,Compound::DB00398,Compound::DB00495,Compound::DB00398,1076,789
2,Compound::DB06803,Synergism,Metabolism,Compound::DB00501,Compound::DB06803,Compound::DB00501,293,715
3,Compound::DB06203,Synergism,Unknown,Compound::DB01050,Compound::DB06203,Compound::DB01050,856,1087
4,Compound::DB01452,Antagonism,Unknown,Compound::DB00652,Compound::DB01452,Compound::DB00652,786,390
...,...,...,...,...,...,...,...,...
346854,Compound::DB00803,No Interaction,No Interaction,Compound::DB00180,Compound::DB00803,Compound::DB00180,281,516
346855,Compound::DB00651,No Interaction,No Interaction,Compound::DB00291,Compound::DB00651,Compound::DB00291,292,337
346856,Compound::DB01231,No Interaction,No Interaction,Compound::DB00822,Compound::DB01231,Compound::DB00822,1048,453
346857,Compound::DB00218,No Interaction,No Interaction,Compound::DB00209,Compound::DB00218,Compound::DB00209,1152,571


# 3. Mapping nhãn và loại các cột thừa

In [19]:
label_map = {
    "No Interaction": -1,
    "Synergism": 0,
    "Antagonism": 1,
    "New Effect": 2
}

In [20]:
def clean_df(df):
    # df = df[df["Pharmacodynamics"] != "No Interaction"].copy()
    df = df.copy()  
    # Map sang label
    df["label"] = df["Pharmacodynamics"].map(label_map)
    # Keep only 3 columns
    return df[["drug1_id", "drug2_id", "label"]].reset_index(drop=True)

In [21]:
train = clean_df(train)
val   = clean_df(val)
test  = clean_df(test)

In [22]:
train.to_csv("train.txt", sep="\t", index=False, header=False)
val.to_csv("val.txt", sep="\t", index=False, header=False)
test.to_csv("test.txt", sep="\t", index=False, header=False)

# Mapping BKG

In [23]:
bkg.head()

,0,1,2
0,1710,1711,0
1,1712,1713,0
2,1714,1715,0
3,1716,1717,0
4,1718,1719,0


In [24]:
import pandas as pd

# Đọc file BKG
bkg = pd.read_csv(
    "raw_data/Drugbank/BKG_file.txt",
    delim_whitespace=True,
    header=None
)

# Giả sử old2new là dict {old_id: new_id}
# Tạo cột mới với mapping
mapped_cols = []
missing_values = set()

for col in [0, 1, 2]:  # áp dụng cho cột 0, 1, 2
    def map_value(x):
        if x in old2new:
            return old2new[x]
        else:
            missing_values.add(x)
            return x  # giữ nguyên nếu không có mapping
    if col == 2:
        mapped_cols.append(bkg[col])
    else:
        mapped_cols.append(bkg[col].map(map_value))

# Tạo DataFrame mới với các cột đã map
bkg_mapped = pd.concat(mapped_cols, axis=1)

# Kiểm tra các giá trị bị thiếu mapping
if missing_values:
    print("Những giá trị không có mapping:", missing_values)
else:
    print("Tất cả giá trị đều có mapping.")

# Lưu ra file txt (tab-separated)
bkg_mapped.to_csv("BKG_file_mapped.txt", sep=" ", index=False, header=False)


C:\Users\Admin\AppData\Local\Temp\ipykernel_28084\621942942.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  bkg = pd.read_csv(


Tất cả giá trị đều có mapping.


In [25]:
bkg["new0"] = bkg[0].map(old2new)
bkg["new1"] = bkg[1].map(old2new)


In [26]:
bkg.head()

,0,1,2,new0,new1
0,1710,1711,0,1508,1509
1,1712,1713,0,1510,1511
2,1714,1715,0,1512,1513
3,1716,1717,0,1514,1515
4,1718,1719,0,1516,1517
